# Single-tool eval: `query_prometheus_range`

Agent loop built from scratch (no `agent.engine` import), bound to `query_prometheus_range` **plus** its metric/label discovery helpers (`list_metric_names`, `get_label_values`) from `agent.tools.prometheus`. Everything outside this backend is still unavailable. This measures how far the model gets on this backend alone.

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage

from agent.config.settings import settings
from agent.llm.client import get_llm
from agent.tools.prometheus import (
    get_label_values,
    list_metric_names,
    query_prometheus_range,
)


In [ ]:
TOOLS = [list_metric_names, get_label_values, query_prometheus_range]
TOOLS_BY_NAME = {t.name: t for t in TOOLS}

SYSTEM_PROMPT = """You are Oracle, a terse infrastructure-operations assistant for the Marconi lab.
You have 3 tools available, all read-only:
- `list_metric_names` — List available Prometheus metric names. Call this FIRST to discover what metrics exist before writing a query. `match` optionally filters by substring.
- `get_label_values` — List values for a Prometheus label such as 'service', 'job', or 'instance'. Use this to discover valid label values before querying.
- `query_prometheus_range` — Run a PromQL range query over the last `minutes`. Use for trends (e.g. CPU over the last 30m). The window is clamped to the configured maximum.
You are read-only. Never suggest you can restart or change anything yourself;
recommend the action for a human operator instead.
Call `list_metric_names` and `get_label_values` FIRST to confirm real values — never guess one in a `query_prometheus_range` query before confirming it exists.
If these tools can't answer the question, say so plainly instead of guessing.
Be concise."""


In [ ]:
def _text(content) -> str:
    """Coerce an LLM message content (str or content-block list) to plain text."""
    if isinstance(content, str):
        return content.strip()
    return "".join(
        b.get("text", "") for b in content if isinstance(b, dict)
    ).strip()


def _accumulate(usage: dict, ai) -> None:
    """Add one LLM turn's token counts (if the endpoint reported any) into usage."""
    reported = ai.usage_metadata or {}
    usage["input"] += reported.get("input_tokens", 0)
    usage["output"] += reported.get("output_tokens", 0)
    usage["total"] += reported.get("total_tokens", 0)


### One turn's tool calls

Multiple tools are bound here, so each call is dispatched by name via `TOOLS_BY_NAME`. The loop handles two things at once: the model choosing between tools, and it calling more than one of them in a single turn.

In [ ]:
async def run_tool_calls(ai) -> list[ToolMessage]:
    """Run every tool call the model made this turn."""
    results = []
    for call in ai.tool_calls:
        print(f"  tool call: {call['name']}({call['args']})")
        try:
            output = await TOOLS_BY_NAME[call["name"]].ainvoke(call["args"])
        except Exception as exc:  # noqa: BLE001 - tool can raise anything
            output = f"(tool call failed: {exc})"
        results.append(ToolMessage(content=str(output), tool_call_id=call["id"]))
    return results


### Turn budget fallback

If the model is still calling tools when `settings.max_tool_steps` runs out, ask it once more, without tools, to summarize whatever it found.

In [ ]:
async def summarize_after_budget(messages: list, usage: dict) -> str:
    messages.append(
        HumanMessage(
            content="You have used the tool budget. Summarize what you found so far, "
            "state the likely cause and next steps, and note what you could not check."
        )
    )
    final = await get_llm().ainvoke(messages)
    _accumulate(usage, final)
    return _text(final.content)


### The loop itself

Ask → if the model answered with no tool calls, return it → otherwise run those tool calls and go around again, up to the turn budget. Every LLM turn's token usage is tallied in `usage` and printed as a one-line transcript once the run ends, so each call below shows what it cost.

In [ ]:
async def run_loop(question: str) -> str:
    messages: list = [SystemMessage(content=SYSTEM_PROMPT), HumanMessage(content=question)]
    llm = get_llm().bind_tools(TOOLS)
    usage = {"input": 0, "output": 0, "total": 0}

    for turn in range(1, settings.max_tool_steps + 1):
        ai = await llm.ainvoke(messages)
        messages.append(ai)
        _accumulate(usage, ai)
        if not ai.tool_calls:
            answer = _text(ai.content)
            break
        messages.extend(await run_tool_calls(ai))
    else:
        turn = settings.max_tool_steps
        answer = await summarize_after_budget(messages, usage)

    print(f"[transcript] {turn} turn(s) — tokens in={usage['input']} out={usage['output']} total={usage['total']}")
    return answer


## In-scope question

A question the bound tools (`list_metric_names`, `get_label_values`, `query_prometheus_range`) together should be able to answer.

In [ ]:
answer = await run_loop("How has the 'up' metric trended over the last 30 minutes?")
print(answer)


## Out-of-scope question

A question that needs evidence outside this backend. Watch whether the model admits the limit or hallucinates past it.

In [ ]:
answer = await run_loop("What Prometheus metrics are available?")
print(answer)
